# Heat Pump Peak Demand Model — Full Review Notebook

**Paper:** *How Much Does Ignoring Occupant Behaviour Cost? Quantifying the Underestimation of Heat Pump Peak Demand in Current Grid Planning Tools*

This notebook runs every stage of the pipeline so you can inspect, verify, and understand the model end-to-end.

### Contents
1. **Setup & Configuration** — Load all parameters
2. **Weather Profiles** — Real EPW data from London Gatwick
3. **Simulation Engine** — Run the 2R1C thermal model (240 runs)
4. **HDD Benchmark** — Heating Degree Day regression
5. **ANOVA Decomposition** — Factor importance (H2)
6. **Interaction Regression** — Nonlinearity test (H3)
7. **Validation vs EoH Field Trial** — COP, peaks, archetypes (730 real homes)
8. **All Figures** — Publication-ready outputs
9. **Hypothesis Verdicts** — Final results

---
## 1. Setup & Configuration

In [ ]:
import sys
from pathlib import Path

# Project root (one level up from notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_palette('colorblind')

print(f'Project root: {PROJECT_ROOT}')
print(f'Python: {sys.version}')

In [ ]:
# Load the YAML configuration
with open(PROJECT_ROOT / 'config' / 'simulation_config.yaml') as f:
    cfg = yaml.safe_load(f)

hp_cfg = cfg['heat_pump']
building_cfg = cfg['building']

print('=== Building ===')
for k, v in building_cfg.items():
    print(f'  {k}: {v}')

print('\n=== Heat Pump ===')
for k, v in hp_cfg.items():
    print(f'  {k}: {v}')

print('\n=== Archetypes ===')
for a_id, a_cfg in cfg['archetypes'].items():
    print(f'  {a_id}: {a_cfg["name"]} — setpoint {a_cfg["setpoint_C"]}C, schedule {a_cfg["schedule_on"]}')

print('\n=== Fabric ===')
for f_id, f_cfg in cfg['fabric'].items():
    print(f'  {f_id}: {f_cfg["name"]} — U_wall={f_cfg["U_wall"]}, ACH={f_cfg["ACH"]}')

print('\n=== Weather Scenarios ===')
for w_id, w_cfg in cfg['weather'].items():
    print(f'  {w_id}: {w_cfg["name"]} — EPW month={w_cfg["epw_month"]}, day={w_cfg["epw_day"]}')

print(f'\n=== Experimental Design ===')
n_cells = len(cfg['archetypes']) * len(cfg['weather']) * len(cfg['fabric'])
n_reps = cfg['replicates']
print(f'  {len(cfg["archetypes"])} archetypes x {len(cfg["weather"])} weather x {len(cfg["fabric"])} fabric = {n_cells} cells')
print(f'  x {n_reps} replicates = {n_cells * n_reps} total runs')
print(f'  x {cfg["timesteps"]} timesteps = {n_cells * n_reps * cfg["timesteps"]:,} output rows')

---
## 2. Weather Profiles (Real EPW Data)

We use London Gatwick TMYx (2009-2023) EPW data from climate.onebuilding.org. Three specific days are extracted to represent mild cold, design cold, and extreme cold conditions.

In [ ]:
from simulation.epw_parser import load_weather_profiles

epw_path = PROJECT_ROOT / cfg['epw_file']
weather_days = {}
for w_id, w_cfg in cfg['weather'].items():
    weather_days[w_id] = {'month': w_cfg['epw_month'], 'day': w_cfg['epw_day']}

profiles = load_weather_profiles(epw_path, weather_days, cfg['dt_minutes'])

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (w_id, wdf) in zip(axes, profiles.items()):
    hours = np.arange(len(wdf)) * cfg['dt_minutes'] / 60
    ax.plot(hours, wdf['T_outdoor_C'], 'b-', linewidth=1.5)
    ax.fill_between(hours, wdf['T_outdoor_C'], alpha=0.15)
    ax.set_title(f"{w_id}: {cfg['weather'][w_id]['name']}\n"
                 f"Mean={wdf['T_outdoor_C'].mean():.1f}C, "
                 f"Min={wdf['T_outdoor_C'].min():.1f}C")
    ax.set_xlabel('Hour')
    ax.grid(alpha=0.3)
    ax.set_xlim(0, 24)
axes[0].set_ylabel('Outdoor Temperature (C)')
fig.suptitle('Real EPW Weather Profiles (London Gatwick TMYx)', fontsize=13)
fig.tight_layout()
plt.show()

---
## 3. Simulation Engine — 2R1C Thermal Model

The core model is a **2R1C lumped-capacitance** building thermal model:

$$C \frac{dT_{in}}{dt} = Q_{hp} + Q_{int} + Q_{solar} - UA \cdot (T_{in} - T_{out})$$

The ASHP model includes:
- COP = 3.5 + 0.12 x T_out (calibrated to RHPP field trial)
- 15% defrost penalty below -2C
- Thermal capacity derating: 2.5%/K below 7C design temp
- Variable-speed modulation based on temperature deficit
- 3 kW auxiliary electric heater for large recovery loads

Running 240 simulations (4 archetypes x 3 weather x 2 fabric x 10 replicates)...

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

from simulation.ochre_runner import run_all_simulations

np.random.seed(cfg['random_seed'])
sim_df = run_all_simulations(cfg, project_root=PROJECT_ROOT)

print(f'Simulation complete: {len(sim_df):,} rows')
print(f'Columns: {list(sim_df.columns)}')
sim_df.head(10)

In [ ]:
# Summary table: peak demand per cell (averaged across replicates)
summary = (
    sim_df.groupby(['archetype', 'weather_scenario', 'fabric', 'replicate'])
    .agg(peak_kW=('electricity_demand_kW', 'max'),
         daily_kWh=('electricity_demand_kW', lambda x: x.sum() * 0.25),
         mean_COP=('COP', 'mean'))
    .groupby(['archetype', 'weather_scenario', 'fabric'])
    .agg(peak_kW=('peak_kW', 'mean'),
         daily_kWh=('daily_kWh', 'mean'),
         mean_COP=('mean_COP', 'mean'))
    .round(2)
    .reset_index()
)

print(f'Summary: {len(summary)} unique cells (4 archetypes x 3 weather x 2 fabric = 24)')
summary

In [ ]:
# Plot demand profiles for all archetypes under W3 (extreme cold), F1
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)

for ax, a_id in zip(axes.flat, ['B1', 'B2', 'B3', 'B4']):
    subset = sim_df[(sim_df['archetype'] == a_id) & 
                    (sim_df['weather_scenario'] == 'W3') & 
                    (sim_df['fabric'] == 'F1') &
                    (sim_df['replicate'] == 0)]
    hours = np.arange(len(subset)) * cfg['dt_minutes'] / 60
    
    ax.plot(hours, subset['electricity_demand_kW'].values, 'C0-', linewidth=1.5, label='Elec demand')
    ax.fill_between(hours, subset['electricity_demand_kW'].values, alpha=0.15, color='C0')
    
    ax2 = ax.twinx()
    ax2.plot(hours, subset['T_indoor_C'].values, 'C1--', linewidth=1, label='T_indoor')
    ax2.plot(hours, subset['T_outdoor_C'].values, 'C2:', linewidth=1, label='T_outdoor')
    ax2.set_ylabel('Temperature (C)')
    
    peak = subset['electricity_demand_kW'].max()
    name = cfg['archetypes'][a_id]['name']
    ax.set_title(f'{a_id}: {name} — Peak = {peak:.1f} kW')
    ax.set_xlabel('Hour')
    ax.set_ylabel('Electricity (kW)')
    ax.grid(alpha=0.3)
    ax.set_xlim(0, 24)

fig.suptitle('Demand Profiles: Extreme Cold (W3), Unimproved Fabric (F1)', fontsize=13)
fig.tight_layout()
plt.show()

### 3b. COP and Thermal Capacity Curves

In [ ]:
from simulation.ochre_runner import compute_cop, compute_thermal_capacity

T_range = np.linspace(-15, 20, 200)
cops = [compute_cop(t, hp_cfg) for t in T_range]
caps = [compute_thermal_capacity(t, hp_cfg) / 1000 for t in T_range]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(T_range, cops, 'C0-', linewidth=2)
ax1.axvline(hp_cfg['defrost_temp_threshold_C'], color='grey', ls=':', alpha=0.5,
            label=f'Defrost threshold ({hp_cfg["defrost_temp_threshold_C"]}C)')
ax1.set_xlabel('Outdoor Temperature (C)')
ax1.set_ylabel('COP')
ax1.set_title('COP vs Outdoor Temperature')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(T_range, caps, 'C1-', linewidth=2)
ax2.axhline(hp_cfg['rated_thermal_capacity_kW'], color='grey', ls='--', alpha=0.5,
            label=f'Rated capacity ({hp_cfg["rated_thermal_capacity_kW"]} kW)')
ax2.axvline(hp_cfg['capacity_design_temp_C'], color='grey', ls=':', alpha=0.5,
            label=f'Design temp ({hp_cfg["capacity_design_temp_C"]}C)')
ax2.set_xlabel('Outdoor Temperature (C)')
ax2.set_ylabel('Thermal Capacity (kW)')
ax2.set_title('Thermal Capacity Derating')
ax2.legend()
ax2.grid(alpha=0.3)

fig.suptitle('ASHP Performance Model (calibrated to RHPP field trial)', fontsize=13)
fig.tight_layout()
plt.show()

---
## 4. HDD Benchmark (H1)

The HDD regression fits `peak_kW = a + b x HDD` to the cross-archetype mean.

**H1:** HDD underestimation of peak demand increases at lower outdoor temperatures.

In [ ]:
from benchmarks.hdd_regression import run_hdd_benchmark

results_dir = PROJECT_ROOT / 'outputs' / 'results_tables'
results_dir.mkdir(parents=True, exist_ok=True)

errors_df, hdd_model = run_hdd_benchmark(sim_df, cfg, results_dir)

print('=== HDD Benchmark Errors ===')
cols = ['archetype', 'weather_scenario', 'fabric', 'peak_kW_actual', 'hdd_peak_kW', 'underestimation_pct']
print(errors_df[cols].to_string(index=False))

print('\n=== Worst-case underestimation by weather ===')
for w in ['W1', 'W2', 'W3']:
    sub = errors_df[errors_df['weather_scenario'] == w]
    worst = sub.loc[sub['underestimation_pct'].idxmax()]
    mean_t = sub['T_mean_outdoor_C'].mean()
    print(f"  {w} ({cfg['weather'][w]['name']}): {worst['underestimation_pct']:.1f}% "
          f"(archetype {worst['archetype']}, fabric {worst['fabric']})")

---
## 5. ANOVA Decomposition (H2)

3-way ANOVA with eta-squared decomposition.

**H2:** Occupant behaviour (archetype) is a stronger predictor of peak demand than building fabric.

In [ ]:
from analytics.anova_decomposition import run_anova

anova_table = run_anova(sim_df, results_dir)

print('=== ANOVA Results ===')
print(anova_table[['sum_sq', 'df', 'F', 'PR(>F)', 'eta_squared']].round(4))

non_resid = anova_table[anova_table.index != 'Residual'].copy()
fig, ax = plt.subplots(figsize=(8, 4))
non_resid['eta_squared'].sort_values().plot(kind='barh', ax=ax, color='C0')
ax.set_xlabel('Eta-squared (proportion of variance explained)')
ax.set_title('ANOVA: Factor Importance for Peak Demand')
ax.grid(alpha=0.3, axis='x')
fig.tight_layout()
plt.show()

arch_eta = anova_table.loc['archetype', 'eta_squared'] if 'archetype' in anova_table.index else 0
fab_eta = anova_table.loc['fabric', 'eta_squared'] if 'fabric' in anova_table.index else 0
print(f'\nH2 test: archetype eta_sq = {arch_eta:.4f} vs fabric eta_sq = {fab_eta:.4f}')
print(f'H2 verdict: {"SUPPORTED" if arch_eta > fab_eta else "NOT SUPPORTED"}')

---
## 6. Interaction Regression & RESET Test (H3)

Compares:
- **Model A** (HDD-linear): peak_kW ~ HDD
- **Model B** (Full interaction): peak_kW ~ T^2 + archetype dummies + fabric + interactions

**H3:** The weather-behaviour interaction is nonlinear.

In [ ]:
from analytics.interaction_regression import run_interaction_regression

reg_results = run_interaction_regression(sim_df, cfg, results_dir)

print('=== Model Comparison ===')
print(reg_results['comparison_df'].to_string(index=False))

print(f'\n=== Ramsey RESET Test ===')
reset = reg_results['reset_test']
print(f'  Statistic: {reset["statistic"]:.4f}')
print(f'  p-value:   {reset["p_value"]:.6f}')
print(f'  H3 verdict: {"SUPPORTED" if reset["p_value"] < 0.05 else "INCONCLUSIVE"}')

---
## 7. Validation Against EoH Field Trial Data

The **Electrification of Heat (EoH)** demonstration project monitored **739 homes** with ASHPs at 30-minute intervals from 2020-2023 (27.7M rows).

We validate:
1. **COP model** — Does our COP curve match real field performance?
2. **Peak demand** — Are simulated peaks realistic?
3. **Behavioural archetypes** — Do real homes cluster into distinct heating patterns?

In [ ]:
# Check if EoH data is available
eoh_path = (PROJECT_ROOT / 'data' / 'external' / 'eoh' / 
            'UKDA-30min-Interval-9209-csv' / 'csv' / 'eoh_cleaned_half_hourly.csv')

if eoh_path.exists():
    print(f'EoH data found at: {eoh_path}')
    print('Loading... (takes ~10s for 27.7M rows)')
    
    from validation.eoh_loader import load_eoh_half_hourly, compute_daily_peaks
    eoh_df = load_eoh_half_hourly(eoh_path)
    daily_peaks = compute_daily_peaks(eoh_df)
    
    print(f'Loaded: {len(eoh_df):,} winter rows, {eoh_df["Property_ID"].nunique()} properties')
    print(f'Daily peaks: {len(daily_peaks):,} property-days')
    EOH_AVAILABLE = True
else:
    print('EoH data not found. Download from UK Data Service (SN 9209).')
    print(f'Expected: {eoh_path}')
    print('Will show pre-computed results instead.')
    EOH_AVAILABLE = False

### 7a. COP Validation

Our model uses compressor COP = 3.5 + 0.12T (from RHPP trial).
The EoH data measures **whole-system COP** (SPF H4 boundary), which is systematically lower.

In [ ]:
import json
from IPython.display import Image, display

val_base = PROJECT_ROOT / 'outputs' / 'validation'

if EOH_AVAILABLE:
    from validation.cop_validation import validate_cop
    cop_stats = validate_cop(eoh_df, hp_cfg, val_base / 'cop')
    print('=== COP Validation ===')
    for k, v in cop_stats.items():
        print(f'  {k}: {v}')
else:
    vs = val_base / 'validation_summary.json'
    if vs.exists():
        with open(vs) as f:
            all_stats = json.load(f)
        print('=== COP Validation (pre-computed) ===')
        for k, v in all_stats.get('cop_validation', {}).items():
            print(f'  {k}: {v}')

cop_fig = val_base / 'cop' / 'fig_cop_validation.png'
if cop_fig.exists():
    display(Image(filename=str(cop_fig), width=700))

### 7b. Peak Demand Validation

In [ ]:
if EOH_AVAILABLE:
    from validation.peak_validation import validate_peaks
    peak_stats = validate_peaks(daily_peaks, sim_df, hp_cfg, val_base / 'peaks')
    print('=== Peak Demand Validation ===')
    for k, v in peak_stats.items():
        print(f'  {k}: {v}')
    
    sim_peaks = sim_df.groupby(['archetype','weather_scenario','fabric','replicate'])['electricity_demand_kW'].max()
    print(f'\nSimulated peak range: {sim_peaks.min():.1f} - {sim_peaks.max():.1f} kW')
    print(f'Real EoH median peak: {peak_stats["real_peak_median_kW"]:.1f} kW')
    print(f'Real EoH P95 peak:    {peak_stats["real_peak_p95_kW"]:.1f} kW')
else:
    if vs.exists():
        print('=== Peak Demand (pre-computed) ===')
        for k, v in all_stats.get('peak_validation', {}).items():
            print(f'  {k}: {v}')

for fname in ['fig_peak_validation.png', 'fig_admd_by_temperature.png']:
    fp = val_base / 'peaks' / fname
    if fp.exists():
        display(Image(filename=str(fp), width=700))

### 7c. Archetype Discovery (k-means Clustering)

Do real homes naturally cluster into distinct heating patterns matching our 4 assumed archetypes?

In [ ]:
arch_csv = val_base / 'archetypes' / 'archetype_cluster_summary.csv'
if arch_csv.exists():
    print('=== Cluster Summary ===')
    print(pd.read_csv(arch_csv).to_string(index=False))

sil_csv = val_base / 'archetypes' / 'silhouette_scores.csv'
if sil_csv.exists():
    print('\n=== Silhouette Scores ===')
    print(pd.read_csv(sil_csv).to_string(index=False))

for fname in ['fig_archetype_centroids.png', 'fig_k4_archetypes.png', 'fig_archetype_heatmap.png']:
    fp = val_base / 'archetypes' / fname
    if fp.exists():
        print(f'\n{fname}:')
        display(Image(filename=str(fp), width=700))

---
## 8. All Publication Figures

In [ ]:
from analytics.demand_surface import build_demand_surfaces
from visualisation.plots import generate_all_figures

archetype_names = {k: v['name'] for k, v in cfg['archetypes'].items()}
surfaces = build_demand_surfaces(sim_df)
figures_dir = PROJECT_ROOT / 'outputs' / 'figures'

generate_all_figures(
    sim_df=sim_df,
    errors_df=errors_df,
    anova_table=anova_table,
    surfaces=surfaces,
    hp_cfg=hp_cfg,
    archetype_names=archetype_names,
    out_dir=figures_dir,
)

print('Figures saved to:', figures_dir)
for f in sorted(figures_dir.glob('*.png')):
    print(f'  {f.name}')

In [ ]:
for fig_path in sorted(figures_dir.glob('*.png')):
    print(f'\n{fig_path.name}:')
    display(Image(filename=str(fig_path), width=700))

---
## 9. Hypothesis Verdicts

In [ ]:
print('=' * 72)
print('  HYPOTHESIS VERDICTS')
print('=' * 72)

# H1
print('\n--- H1: HDD underestimation increases at lower outdoor temperatures ---')
max_errors = []
for w in ['W1', 'W2', 'W3']:
    sub = errors_df[errors_df['weather_scenario'] == w]
    worst = sub.loc[sub['underestimation_pct'].idxmax()]
    mean_t = sub['T_mean_outdoor_C'].mean()
    max_err = sub['underestimation_pct'].max()
    max_errors.append(max_err)
    print(f"  {w} ({cfg['weather'][w]['name']}): T_mean={mean_t:+.1f}C -> "
          f"max underestimation = {max_err:.1f}% (archetype {worst['archetype']})")
h1 = all(max_errors[i] <= max_errors[i+1] for i in range(len(max_errors)-1))
print(f'  -> H1: {"SUPPORTED" if h1 else "NOT SUPPORTED"}')

# H2
print('\n--- H2: Occupant behaviour > fabric for peak demand ---')
arch_eta = anova_table.loc['archetype', 'eta_squared'] if 'archetype' in anova_table.index else 0
fab_eta = anova_table.loc['fabric', 'eta_squared'] if 'fabric' in anova_table.index else 0
print(f'  archetype eta_sq = {arch_eta:.4f}')
print(f'  fabric eta_sq    = {fab_eta:.4f}')
h2 = arch_eta > fab_eta
print(f'  -> H2: {"SUPPORTED" if h2 else "NOT SUPPORTED"}')

# H3
print('\n--- H3: Nonlinear weather-behaviour interaction ---')
reset = reg_results['reset_test']
comp = reg_results['comparison_df']
print(f'  RESET test p-value: {reset["p_value"]:.6f}')
print(f'  Model A R2: {comp.iloc[0]["R_squared"]:.4f}  |  Model B R2: {comp.iloc[1]["R_squared"]:.4f}')
h3 = reset['p_value'] < 0.05
print(f'  -> H3: {"SUPPORTED" if h3 else "INCONCLUSIVE"}')

print('\n' + '=' * 72)
print(f'  H1 (HDD underestimates more in cold):       {"SUPPORTED" if h1 else "NOT SUPPORTED"}')
print(f'  H2 (Behaviour > Fabric for peak demand):    {"SUPPORTED" if h2 else "NOT SUPPORTED"}')
print(f'  H3 (Nonlinear interaction, HDD inadequate): {"SUPPORTED" if h3 else "INCONCLUSIVE"}')
print('=' * 72)

---
## 10. Test Suite

In [ ]:
import subprocess
result = subprocess.run(
    [sys.executable, '-m', 'pytest', str(PROJECT_ROOT / 'tests'), '-v', '--tb=short'],
    capture_output=True, text=True, cwd=str(PROJECT_ROOT)
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)